# Canonical 200k Representative Clustering

Sparse spectral clustering with AMG on the 100k-note / 200k-user analysis
slice, followed by Method-A and Method-B reassignment diagnostics. Method B is
the canonical assignment consumed downstream.

Outputs are written to `data/interim/`; smoke outputs are isolated under
`.artifacts/smoke/data/interim/`.


In [ ]:
import os
os.environ.setdefault("PYTHONHASHSEED", "42")
import numpy as np
import random

np.random.seed(42)
random.seed(42)
EIGEN_SOLVER = "amg"


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "config.txt").exists():
            return path
    raise RuntimeError("config.txt not found")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from src.spectral_fast import SpectralFastConfig, run_clustering_spectral_fast
from src.clustering import build_user_cluster_summary
from src.io import (
    attach_note_metadata,
    get_config_float,
    get_config_int,
    get_data_root,
    get_interim_dir,
    get_test_mode,
    load_clustering_slice,
    load_note_metadata,
    save_table,
)

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 1000)
sns.set_theme(style='whitegrid', context='notebook')

TEST_MODE = get_test_mode()
print(f'=== TEST_MODE = {TEST_MODE} ===')
print(f'    PROJECT_ROOT = {PROJECT_ROOT}')


## Configuration

In [ ]:
config = SpectralFastConfig(
    target_note_count=500 if TEST_MODE else get_config_int('FULL_CLUSTER_TARGET_NOTES', 100000),
    target_user_count=1000 if TEST_MODE else get_config_int('FULL_CLUSTER_TARGET_USERS', 200000),
    min_note_ratings=get_config_int('FULL_CLUSTER_MIN_NOTE_RATINGS', 3),
    hours_window=48,
    k_min=2,
    k_max=4 if TEST_MODE else get_config_int('FULL_CLUSTER_K_MAX', 7),
    random_state=42,
    stability_runs=2 if TEST_MODE else get_config_int('FULL_CLUSTER_STABILITY_RUNS', 5),
    stability_subsample_frac=get_config_float('FULL_CLUSTER_STABILITY_SUBSAMPLE_FRAC', 0.8),
    knn_neighbors=get_config_int('FULL_CLUSTER_KNN_NEIGHBORS', 15),
    silhouette_sample_size=5000,
    ann_backend_min_users=8000,
    pynndescent_n_trees=32,
    pynndescent_max_candidates=60,
    require_ann=not TEST_MODE,
    kmeans_n_init=10,
    eigen_solver='arpack' if TEST_MODE else EIGEN_SOLVER,
)

MASTER_PARQUET = get_data_root() / 'master_full.parquet'
INTERIM_DIR = get_interim_dir()
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(f'MASTER_PARQUET: {MASTER_PARQUET}')
print(f'INTERIM_DIR: {INTERIM_DIR}')
print(config)


## Load and Filter Ratings

In [ ]:
ratings = load_clustering_slice(
    MASTER_PARQUET,
    target_note_count=config.target_note_count,
    target_user_count=config.target_user_count,
    hours_window=config.hours_window,
    min_note_ratings=config.min_note_ratings,
    batch_size=get_config_int('FULL_CLUSTER_BATCH_SIZE', 250_000),
)
print(f'Rows: {len(ratings):,}  Notes: {ratings["noteId"].nunique():,}  Users: {ratings["raterParticipantId"].nunique():,}')

## Run AMG Clustering

In [ ]:
cluster_result = run_clustering_spectral_fast(ratings, config=config)

sil_df = cluster_result['silhouette_table']
stab_df = cluster_result['stability_table']
graph_df = cluster_result['graph_diagnostics']
runtime_df = cluster_result['runtime_table']
user_cluster_df = cluster_result['user_cluster_df']
ratings_clustered = cluster_result['df_clustered']
ratings_filtered = cluster_result['df_sub']
embedding_kmax = cluster_result['embedding']
chosen_k = cluster_result['n_clusters']

note_metadata = load_note_metadata(PROJECT_ROOT / 'raw', ratings_filtered['noteId'].unique())
ratings_filtered = attach_note_metadata(ratings_filtered, note_metadata)
ratings_clustered = attach_note_metadata(ratings_clustered, note_metadata)

print(f'Chosen k: {chosen_k}')
print(f'Embedding shape: {embedding_kmax.shape}')
display(stab_df)
display(sil_df)
display(user_cluster_df.head())

## Save Standard Outputs + Embedding

In [ ]:
# Save standard outputs (algorithm-chosen labels)
save_table(ratings_filtered, INTERIM_DIR / 'ratings_filtered.parquet')
save_table(ratings_clustered, INTERIM_DIR / 'ratings_clustered.parquet')
save_table(user_cluster_df, INTERIM_DIR / 'user_clusters.parquet')
save_table(sil_df, INTERIM_DIR / 'silhouette_over_k.parquet')
save_table(stab_df, INTERIM_DIR / 'stability_over_k.parquet')
save_table(graph_df, INTERIM_DIR / 'graph_diagnostics.parquet')
save_table(runtime_df, INTERIM_DIR / 'runtime_diagnostics.parquet')

# Save embedding (NEW for reassigned variants)
emb_cols = [f'dim_{i}' for i in range(embedding_kmax.shape[1])]
embedding_df = pd.DataFrame(embedding_kmax, columns=emb_cols)
embedding_df.insert(0, 'raterParticipantId', user_cluster_df['raterParticipantId'].values)
save_table(embedding_df, INTERIM_DIR / 'embedding.parquet')
print(f'Saved embedding: {embedding_df.shape}')

## Outlier Reassignment (Method A + Method B)

In [ ]:
# === REASSIGNMENT POST-PROCESSING ===
# Identify outlier clusters: <1% of total users
cluster_sizes = user_cluster_df['cluster'].value_counts().sort_index()
total_users = len(user_cluster_df)
threshold = max(total_users * 0.01, 500)
main_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
outlier_clusters = cluster_sizes[cluster_sizes < threshold].index.tolist()

print(f'Total users: {total_users:,}')
print(f'Cluster sizes: {cluster_sizes.to_dict()}')
print(f'Main clusters (>={threshold:.0f} users): {main_clusters}')
print(f'Outlier clusters (to reassign): {outlier_clusters}')

if not outlier_clusters:
    print("No outliers detected — reassignment is a no-op.")
    save_table(user_cluster_df.copy(), INTERIM_DIR / 'user_clusters_method_a_embedding.parquet')
    save_table(user_cluster_df.copy(), INTERIM_DIR / 'user_clusters_method_b_voteprofile.parquet')
elif len(main_clusters) < 2:
    print("Fewer than 2 main clusters — reassignment is a no-op.")
    save_table(user_cluster_df.copy(), INTERIM_DIR / 'user_clusters_method_a_embedding.parquet')
    save_table(user_cluster_df.copy(), INTERIM_DIR / 'user_clusters_method_b_voteprofile.parquet')
else:
    # ===== METHOD A: Embedding centroid distance =====
    print("\n--- METHOD A: Embedding centroid distance ---")
    user_to_emb_idx = {uid: i for i, uid in enumerate(user_cluster_df['raterParticipantId'].values)}
    emb_k = embedding_kmax[:, :chosen_k]
    main_centroids = {}
    for c in main_clusters:
        mask = (user_cluster_df['cluster'] == c).values
        main_centroids[c] = emb_k[mask].mean(axis=0)
        print(f'  Main cluster {c} centroid shape={main_centroids[c].shape} from {mask.sum()} users')
    
    labels_method_a = user_cluster_df['cluster'].values.copy()
    for outlier_c in outlier_clusters:
        mask = (user_cluster_df['cluster'] == outlier_c).values
        outlier_emb = emb_k[mask]
        distance_columns = []
        for main_c in main_clusters:
            d = np.linalg.norm(outlier_emb - main_centroids[main_c], axis=1)
            distance_columns.append(d)
        distance_matrix = np.column_stack(distance_columns)
        nearest_idx = distance_matrix.argmin(axis=1)
        new_labels = np.array([main_clusters[i] for i in nearest_idx])
        labels_method_a[mask] = new_labels
        unique, counts = np.unique(new_labels, return_counts=True)
        print(f'  Outlier cluster {outlier_c} ({mask.sum()} users) reassigned: {dict(zip(unique.tolist(), counts.tolist()))}')
    
    # ===== METHOD B: Vote-profile correlation =====
    print("\n--- METHOD B: Vote-profile correlation ---")
    # Compute per-main-cluster average vote profile
    main_user_ids = user_cluster_df[user_cluster_df['cluster'].isin(main_clusters)][['raterParticipantId', 'cluster']]
    main_ratings = ratings_clustered.merge(main_user_ids, on='raterParticipantId', suffixes=('', '_y'))
    # main_ratings already has cluster column from the merge
    note_profile = main_ratings.groupby(['noteId', 'cluster'])['vote'].mean().unstack(fill_value=np.nan)
    print(f'  Note profile shape: {note_profile.shape}, columns (main clusters): {note_profile.columns.tolist()}')
    
    outlier_user_ids = user_cluster_df[user_cluster_df['cluster'].isin(outlier_clusters)]['raterParticipantId'].values
    print(f'  Computing vote-profile correlation for {len(outlier_user_ids)} outlier users...')
    
    labels_method_b = user_cluster_df['cluster'].values.copy()
    user_idx_in_df = {uid: i for i, uid in enumerate(user_cluster_df['raterParticipantId'].values)}
    
    outlier_ratings = ratings_clustered[ratings_clustered['raterParticipantId'].isin(outlier_user_ids)]
    outlier_ratings_grouped = outlier_ratings.groupby('raterParticipantId')
    
    reassign_log = {c: 0 for c in main_clusters}
    skipped = 0
    for uid in outlier_user_ids:
        if uid not in outlier_ratings_grouped.groups:
            skipped += 1
            continue
        user_votes_df = outlier_ratings_grouped.get_group(uid).set_index('noteId')['vote'].astype(float)
        correlations = {}
        for c in main_clusters:
            if c not in note_profile.columns:
                continue
            cluster_profile = note_profile[c].dropna()
            common = user_votes_df.index.intersection(cluster_profile.index)
            if len(common) >= 5:
                try:
                    corr = user_votes_df.loc[common].corr(cluster_profile.loc[common])
                    if not pd.isna(corr):
                        correlations[c] = corr
                except Exception:
                    pass
        if correlations:
            best_cluster = max(correlations, key=correlations.get)
            labels_method_b[user_idx_in_df[uid]] = best_cluster
            reassign_log[best_cluster] += 1
        else:
            skipped += 1
    print(f'  Method B reassignment distribution: {reassign_log}')
    print(f'  Skipped (insufficient overlap): {skipped}')
    
    # Save both reassignments
    user_clusters_a = user_cluster_df.copy()
    user_clusters_a['cluster'] = labels_method_a
    save_table(user_clusters_a, INTERIM_DIR / 'user_clusters_method_a_embedding.parquet')
    
    user_clusters_b = user_cluster_df.copy()
    user_clusters_b['cluster'] = labels_method_b
    save_table(user_clusters_b, INTERIM_DIR / 'user_clusters_method_b_voteprofile.parquet')
    
    # Compare A vs B agreement
    agreement = (labels_method_a == labels_method_b).mean()
    print(f'\n--- COMPARISON ---')
    print(f'  Method A vs Method B agreement (entire population): {agreement:.4f}')
    outlier_mask = user_cluster_df['cluster'].isin(outlier_clusters).values
    outlier_agreement = (labels_method_a[outlier_mask] == labels_method_b[outlier_mask]).mean()
    print(f'  Method A vs Method B agreement (outlier users only): {outlier_agreement:.4f}')
    print(f'  Cluster sizes after Method A: {pd.Series(labels_method_a).value_counts().sort_index().to_dict()}')
    print(f'  Cluster sizes after Method B: {pd.Series(labels_method_b).value_counts().sort_index().to_dict()}')

## User/Cluster Summary

In [ ]:
# Per-cluster user/vote stats for original + reassigned labels
user_stats_orig, cluster_summary_orig = build_user_cluster_summary(ratings_clustered)
save_table(user_stats_orig, INTERIM_DIR / 'user_stats.parquet')
save_table(cluster_summary_orig, INTERIM_DIR / 'cluster_summary.parquet')
display(cluster_summary_orig)